# 02 — Transformación de JSON a CSV

Este notebook no llama a ninguna API. Lee las respuestas crudas guardadas en `data/raw/`, construye una fila limpia por película y exporta el resultado a `data/processed/peliculas.csv`. Así cualquier integrante puede reproducir esta etapa sin tener las API keys.

In [1]:
from pathlib import Path
import json

import pandas as pd


In [2]:
def encontrar_raiz_proyecto(inicio: Path) -> Path:
    for carpeta in (inicio, *inicio.parents):
        if (carpeta / '.env.example').exists():
            return carpeta
    raise FileNotFoundError('No se encontró .env.example en las carpetas superiores.')

def cargar_json_mas_reciente(carpeta: Path) -> tuple[dict, Path]:
    archivos = list(carpeta.glob('*.json'))
    if not archivos:
        raise FileNotFoundError(f'No se encontraron JSON en {carpeta}')
    archivo = max(archivos, key=lambda ruta: ruta.stat().st_mtime)
    with archivo.open(encoding='utf-8') as contenido:
        return json.load(contenido), archivo

RAIZ_PROYECTO = encontrar_raiz_proyecto(Path.cwd().resolve())
detalles_tmdb, archivo_tmdb = cargar_json_mas_reciente(RAIZ_PROYECTO / 'data' / 'raw' / 'tmdb')
detalles_omdb, archivo_omdb = cargar_json_mas_reciente(RAIZ_PROYECTO / 'data' / 'raw' / 'omdb')

print(f'TMDB: {archivo_tmdb.relative_to(RAIZ_PROYECTO)}')
print(f'OMDb: {archivo_omdb.relative_to(RAIZ_PROYECTO)}')

TMDB: data\raw\tmdb\minecraft_movie_2026-11-10.json
OMDb: data\raw\omdb\minecraft_movie_2026-11-10.json


In [3]:
calificaciones_omdb = {
    calificacion['Source']: calificacion['Value']
    for calificacion in detalles_omdb.get('Ratings', [])
}

fila_pelicula = {
    'tmdb_id': detalles_tmdb.get('id'),
    'imdb_id': detalles_tmdb.get('imdb_id') or detalles_omdb.get('imdbID'),
    'titulo': detalles_tmdb.get('title'),
    'titulo_original': detalles_tmdb.get('original_title'),
    'fecha_estreno': detalles_tmdb.get('release_date'),
    'presupuesto_usd': detalles_tmdb.get('budget'),
    'recaudacion_usd': detalles_tmdb.get('revenue'),
    'generos': ' | '.join(genero['name'] for genero in detalles_tmdb.get('genres', [])),
    'popularidad_tmdb': detalles_tmdb.get('popularity'),
    'rating_tmdb': detalles_tmdb.get('vote_average'),
    'votos_tmdb': detalles_tmdb.get('vote_count'),
    'rating_imdb': detalles_omdb.get('imdbRating'),
    'votos_imdb': detalles_omdb.get('imdbVotes'),
    'metascore': detalles_omdb.get('Metascore'),
    'rating_rotten_tomatoes': calificaciones_omdb.get('Rotten Tomatoes'),
    'taquilla_omdb': detalles_omdb.get('BoxOffice'),
    'fuente_tmdb': archivo_tmdb.name,
    'fuente_omdb': archivo_omdb.name,
}

peliculas = pd.DataFrame([fila_pelicula])
peliculas['fecha_estreno'] = pd.to_datetime(peliculas['fecha_estreno'], errors='coerce')

columnas_numericas = ['presupuesto_usd', 'recaudacion_usd', 'popularidad_tmdb', 'rating_tmdb', 'votos_tmdb', 'rating_imdb', 'metascore']
for columna in columnas_numericas:
    peliculas[columna] = pd.to_numeric(peliculas[columna], errors='coerce')

peliculas

,tmdb_id,imdb_id,titulo,titulo_original,fecha_estreno,presupuesto_usd,recaudacion_usd,generos,popularidad_tmdb,rating_tmdb,votos_tmdb,rating_imdb,votos_imdb,metascore,rating_rotten_tomatoes,taquilla_omdb,fuente_tmdb,fuente_omdb
0,950387,tt3566834,Una película de Minecraft,A Minecraft Movie,2025-03-31,150000000,960387780,Familia | Fantasía | Comedia | Aventura,18.9597,6.192,3440,5.6,"168,568",45,None,"$424,087,780",minecraft_movie_2026-11-10.json,minecraft_movie_2026-11-10.json


In [4]:
ruta_csv = RAIZ_PROYECTO / 'data' / 'processed' / 'peliculas.csv'
ruta_csv.parent.mkdir(parents=True, exist_ok=True)
peliculas.to_csv(ruta_csv, index=False, encoding='utf-8')

print(f'CSV creado en: {ruta_csv.relative_to(RAIZ_PROYECTO)}')
print(f'Filas: {peliculas.shape[0]} | Columnas: {peliculas.shape[1]}')
print(f"Duplicados por tmdb_id: {peliculas.duplicated(subset='tmdb_id').sum()}")
print('Nulos por columna:')
peliculas.isna().sum().sort_values(ascending=False)

CSV creado en: data\processed\peliculas.csv
Filas: 1 | Columnas: 18
Duplicados por tmdb_id: 0
Nulos por columna:


rating_rotten_tomatoes    1
tmdb_id                   0
imdb_id                   0
titulo                    0
fecha_estreno             0
titulo_original           0
recaudacion_usd           0
generos                   0
popularidad_tmdb          0
presupuesto_usd           0
rating_tmdb               0
votos_tmdb                0
votos_imdb                0
rating_imdb               0
metascore                 0
taquilla_omdb             0
fuente_tmdb               0
fuente_omdb               0
dtype: int64

## Interpretación

Por ahora el CSV tiene una sola fila porque se extrajo una sola película. Eso es correcto para comprobar el pipeline, pero no alcanza para comparar ni cumplir un futuro requisito de volumen. Cuando se incorporen comparables, cada película debe agregarse como una nueva fila usando esta misma estructura de columnas.